# الدرس 01 - مقدمة لوكلاء الذكاء الاصطناعي

مرحبًا بك في الدرس الأول من دورة **وكلاء الذكاء الاصطناعي للمبتدئين**!

الوكيل **AI agent** هو برنامج يستخدم نموذج لغة كبير (LLM) كمحرك تفكير له ويمكنه اتخاذ *إجراءات* في العالم الحقيقي — مثل استدعاء واجهات برمجة التطبيقات، أو الاستعلام من قواعد البيانات، أو تشغيل التعليمات البرمجية — لتحقيق هدف نيابة عن المستخدم.

في هذا الدفتر ستُنشئ وكيلك الأول: وهو **وكيل السفر** الذي يوصي بوجهات عطلات. على طول الطريق ستتعلم كيفية:

1. الاتصال بخدمة Azure AI Foundry Agent باستخدام **إطار عمل مايكروسوفت للوكلاء**.
2. إعطاء الوكيل **أداة** — وهي دالة بايثون بسيطة يمكنه استدعاؤها.
3. تشغيل الوكيل وفحص استجابته.
4. بث استجابة الوكيل رمزًا برمز.


## الإعداد

قبل تشغيل دفتر الملاحظات هذا، تأكد من أنك:

1. **تمتلك مشروع Azure AI Foundry** مع نموذج دردشة منشور (مثل `gpt-4o-mini`).
2. **قد سجلت الدخول باستخدام Azure CLI** — قم بتشغيل `az login` في الطرفية الخاصة بك.
3. **قمت بتعيين متغيرات البيئة المطلوبة:**
   - `AZURE_AI_PROJECT_ENDPOINT` — نقطة نهاية مشروع Azure AI Foundry الخاص بك.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — اسم النموذج المنشور الخاص بك.

الخلية أدناه تثبت حزم بايثون التي تحتاجها.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## إنشاء وكيلك الأول

يحتاج الوكيل إلى شيئين:

- **تعليمات** تخبره *من هو* و*كيف يتصرف* (موجه النظام).
- **أدوات** — دوال بايثون مزينة بـ `@tool` يمكن للوكيل استدعاؤها لاسترجاع المعلومات أو أداء الإجراءات.

فيما يلي نعرّف أداة بسيطة تُرجع قائمة بالأماكن السياحية الشهيرة. سيستخدم الوكيل هذه الأداة عندما يطلب المستخدم توصيات سفر.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## الاستجابات المتدفقة

لتجربة أكثر تفاعلية يمكنك **بث** رد الوكيل. بدلاً من الانتظار للرد الكامل، يقدم الوكيل أجزاء من النص أثناء توليدها. هذا مفيد بشكل خاص في واجهات الدردشة حيث ترغب في عرض المخرجات في الوقت الفعلي.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## الملخص

في هذا الدرس تعلمت كيفية:

- **إنشاء مزود** يتصل بخدمة Azure AI Foundry Agent عبر `AzureAIProjectAgentProvider`.
- **تعريف أداة** باستخدام المزخرف `@tool` بحيث يمكن للوكيل استدعاء وظائف البايثون الخاصة بك.
- **تشغيل الوكيل** مع رسالة من المستخدم وعرض استجابته.
- **بث الاستجابات** للإخراج في الوقت الحقيقي.

في الدرس التالي سنستكشف أُطُر الوكلاء بمزيد من العمق ونتعلم كيفية تزويد الوكلاء بأدوات أكثر قوة وقدرات على التفكير متعدد الخطوات.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**إخلاء مسؤولية**:  
تمت ترجمة هذا المستند باستخدام خدمة الترجمة الآلية [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى جاهدين لتحقيق الدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر المعتمد. بالنسبة للمعلومات الحيوية، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسيرات خاطئة ناتجة عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
